In [ ]:
import os
from dotenv import load_dotenv


my_key = os.getenv("APIKEY")

if not my_key:
    raise ValueError("Error: APIKEY not found in .env file")

# S&P Market Predictor: Machine Learning & Macro Analysis.

This project aims to predict daily movements of the S&P index by combining historical price action with macroeconomic data from Fed.Res.
We transition from a technical model to an advanced one that incorporates Inflation, Interest Rates and GDP.

## 1. Multi-Source Data Acquisition.
We fetch high-quality financial data from 2 main sources:

- **Yahoo Finance (yfinance)**: for historical S&P 500 OHLCV data.
- **FRED API (Federal Reserve)**: to integrate key economic indicators

In [ ]:
import yfinance as yf
import pandas as pd
import os

In [ ]:
if os.path.exists("sp500.csv"):
    sp500 = pd.read_csv("sp500.csv", index_col=0)
else:
    sp500 = yf.Ticker("^GSPC")
    sp500 = sp500.history(period="max")
    sp500.to_csv("sp500.csv")

In [ ]:
sp500.index = pd.to_datetime(sp500.index)

In [ ]:
sp500

In [ ]:
sp500.plot.line(y="Close", use_index=True)

In [ ]:
del sp500["Dividends"]
del sp500["Stock Splits"]

## 2. Target definition and Data Preparation.

To train our model, we create a 'Target' variable. 
This is a binary indicator (0 or 1) that tells the model if the S&P 500 price will be higher (1) or lower (0) on the following day.

In [ ]:
sp500["Tomorrow"] = sp500["Close"].shift(-1)

In [ ]:
sp500["Target"] = (sp500["Tomorrow"] > sp500["Close"]).astype(int)

In [ ]:
sp500.index = pd.to_datetime(sp500.index, utc=True).tz_localize(None)
sp500 = sp500.loc["1990-01-01":].copy()

## 3 . Machine Learning Strategy: Random Forest.

In this section, we initialize the 'Random Forest Classifier'.
We choose this model for its ability to handle non-linear relationships and its robustness against overfitting in financial time-series data.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, min_samples_split=100, random_state=1)

train = sp500.iloc[:-100]
test = sp500.iloc[-100:]

predictors = ["Close", "Volume", "Open", "High", "Low"]
model.fit(train[predictors], train["Target"])

In [ ]:
from sklearn.metrics import precision_score

preds = model.predict(test[predictors])
preds = pd.Series(preds, index=test.index)
precision_score(test["Target"], preds)

In [ ]:
combined = pd.concat([test["Target"], preds], axis=1)
combined.plot()

In [ ]:
def predict(train, test, predictors, model):
    model.fit(train[predictors], train["Target"])
    preds = model.predict(test[predictors])
    preds = pd.Series(preds, index=test.index, name="Predictions")
    combined = pd.concat([test["Target"], preds], axis=1)
    return combined

In [ ]:
def backtest(data, model, predictors, start=2500, step=250):
    all_predictions = []

    for i in range(start, data.shape[0], step):
        train = data.iloc[max(0, i-2500):i].copy()
        test = data.iloc[i:(i+step)].copy()
        predictions = predict(train, test, predictors, model)
        all_predictions.append(predictions)
    
    return pd.concat(all_predictions)

In [ ]:
predictions = backtest(sp500, model, predictors)

In [ ]:
predictions["Predictions"].value_counts()

In [ ]:
precision_score(predictions["Target"], predictions["Predictions"])

In [ ]:
predictions["Target"].value_counts() / predictions.shape[0]

In [ ]:
horizons = [2, 5, 60, 250, 1000]
new_predictors = []

for horizon in horizons:
    rolling_averages = sp500.rolling(horizon).mean()

    ratio_column = f"Close_Ratio_{horizon}"
    sp500[ratio_column] = sp500["Close"] / rolling_averages["Close"]

    trend_column = f"Trend_{horizon}"
    sp500[trend_column] = sp500.shift(1).rolling(horizon).sum()["Target"]

    new_predictors+= [ratio_column, trend_column]

In [ ]:
sp500 = sp500.dropna(subset=sp500.columns[sp500.columns != "Tomorrow"])

In [ ]:
sp500

In [ ]:
model = RandomForestClassifier(n_estimators=200, min_samples_split=50, random_state=1)

In [ ]:
def predict(train, test, predictors, model):
    model.fit(train[predictors], train["Target"])
    preds = model.predict_proba(test[predictors])[:,1]
    preds[preds >=.6] = 1
    preds[preds <.6] = 0
    preds = pd.Series(preds, index=test.index, name="Predictions")
    combined = pd.concat([test["Target"], preds], axis=1)
    return combined

In [ ]:
predictions = backtest(sp500, model, new_predictors)

In [ ]:
predictions["Predictions"].value_counts()

In [ ]:
precision_score(predictions["Target"], predictions["Predictions"])

In [ ]:
predictions["Target"].value_counts() / predictions.shape[0]

In [ ]:
predictions

In [ ]:
!pip install fredapi

In [ ]:
from fredapi import Fred
import pandas as pd
fred = Fred(api_key=my_key)

In [ ]:
interest_rates = fred.get_series('FEDFUNDS')
inflation = fred.get_series('CPIAUCSL')
gdp = fred.get_series('GDPC1')

## 4 . Merging S&P500 price data with Interest Rates, Inflation, and GPD to improve model context.

In [ ]:
m_cols = ['Interest_Rate', 'Inflation', 'GDP']

to_drop = [c for c in sp500.columns if any(x in c for x in ['_x', '_y'] + m_cols)]
sp500 = sp500.drop(columns=to_drop, errors='ignore')

m_df = pd.DataFrame({
    'Interest_Rate': interest_rates,
    'Inflation': inflation,
    'GDP': gdp
    
})

sp500.index = pd.to_datetime(sp500.index).date
m_df.index = pd.to_datetime(m_df.index).date

sp500 = sp500.merge(m_df, left_index=True, right_index=True, how='left')

sp500[m_cols] = sp500[m_cols].ffill().shift(1)

sp500 = sp500.dropna()

print("Dataset updated successfully!")
print(sp500.tail())

In [ ]:
old_predictors = ["Close", "Volume", "Open", "High", "Low", "Close_Ratio_2", "Trend_2", "Close_Ratio_5", "Trend_5", "Close_Ratio_60",
                  "Trend_60", "Close_Ratio_250", "Trend_250", "Close_Ratio_1000", "Trend_1000"]

macro_predictors = ["Interest_Rate", "Inflation", "GDP"]

new_predictors = old_predictors + macro_predictors

In [ ]:
predictions = backtest(sp500, model, new_predictors)

In [ ]:
from sklearn.metrics import precision_score
score = precision_score(predictions["Target"], predictions["Predictions"])
print(f"New precision score is:{score:.2%}") 

In [ ]:
train = sp500.iloc[:-100]
test = sp500.iloc[-100:]

from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=200, min_samples_split=50, random_state=1)

model.fit(train[new_predictors], train["Target"])

print(f"Model successfully trained on {len(train)} days using {len(new_predictors)} predictors!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score

sp500.columns = sp500.columns.str.strip()
new_predictors = [str(c).strip() for c in new_predictors]

if len(new_predictors) != 18:
    print (f"Attention: The list has {len(new_predictors)} elements instead of 18.")

else:
    try:

        importances = pd.Series(model.feature_importances_, index=new_predictors)
        importances.sort_values(ascending=False).plot(kind='bar', figsize=(12, 5))
        plt.title("What factors influence the model the most?")
        plt.ylabel("Importance Score")
        plt.show()

        probs = model.predict_proba(sp500[new_predictors])[:,1]
        custom_preds = (probs >= 0.6).astype(int)

        score = precision_score(sp500["Target"], custom_preds)
        print(f"___MISSION COMPLETED___")
        print(f"New Precision Score (60% threshold): {score:.2%}")
        print(f"Days when the model bet: {sum(custom_preds)}")

    except Exception as e:
        print(f"Technical Error: {e}")
    

In [ ]:
train = sp500.iloc[:-1000]
test = sp500.iloc[-1000:]

probs = model.predict_proba(test[new_predictors])[:,1]
custom_preds = (probs >= 0.6).astype(int)

from sklearn.metrics import precision_score
real_score = precision_score(test["Target"], custom_preds)

print(f"___RWP 'Real World Performance'___")
print(f"Real Precision (Never seen Data): {real_score:.2%}")
print(f"Transactions made in the last 1000 days: {sum(custom_preds)}")


## 5 . The 'Truth-Test': Out-of-Sample Validation
After achieving high initial scores, we perform a strict out-of-sample test. We train the model on historical data and test it on the most 1000 days-data it has never seen before.

We also optimize the 'Probability Threshold to 0.55/0.60, making the model "conservative"- it only trades when it has high confidence.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
fresh_model = RandomForestClassifier(n_estimators=200, min_samples_split=50, random_state=1)

train = sp500.iloc[:-1000]
test = sp500.iloc[-1000:]

fresh_model.fit(train[new_predictors], train["Target"])
probs = fresh_model.predict_proba(test[new_predictors])[:,1]
custom_preds = (probs >= 0.55).astype(int)

from sklearn.metrics import precision_score
real_score = precision_score(test["Target"], custom_preds)

print(f"Truth Test")
print(f"RWP: {real_score:.2%}")
print(f"Transactions made in 1000 days: {sum(custom_preds)}")

    

## 6 . Final Results, Performance and Considerations:
The integration of Macroeconomic features successfully pushed the model's 'real-world precision' to 58.62%.

Insight: Interest Rates and Inflation proved to be higher-impact predictors than several short-term technical indicators.

Conclusion: This demonstrate that combining fundamental economic cycles with technical price action provides a significant statistical edge.